# 06_Inference
**발표 스킬 자동 분석 시스템 - 추론 파이프라인**

```
음성 파일 → Whisper STT → 전처리 → 모델 추론 → 결과 출력
```

| 셀 | 내용 |
|---|---|
| 셀 1 | Drive 마운트 + Whisper/ffmpeg 설치 |
| 셀 2 | 학습된 모델 로드 (05_feature_concat과 동일 구조) |
| 셀 3 | 스케일러 복원 (train.csv 기반) |
| 셀 4 | Whisper STT + feature 추출 함수 |
| 셀 5 | 추론 + 피드백 출력 함수 |
| 셀 6 | **음성 파일 입력** → 분석 실행 |
| 셀 7 | **텍스트 직접 입력** → 분석 실행 (음성 없을 때) |

## 셀 1 — Drive 마운트 + 패키지 설치

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

# Whisper 및 의존 패키지 설치
!pip install openai-whisper -q
!apt-get install -y ffmpeg -q
!pip install konlpy -q   # 한국어 형태소 분석 (어휘 다양성)

## 셀 2 — 모델 로드 (05_feature_concat과 동일 구조)

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

# ── 경로 설정 ──────────────────────────────────────────
MODEL_PATH = '/content/drive/MyDrive/presentation_data/best_model_concat.pt'
TRAIN_CSV  = '/content/drive/MyDrive/presentation_data/train.csv'
# ──────────────────────────────────────────────────────

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')


class RoBERTaWithFeatures(nn.Module):
    def __init__(self, model_name, num_features=5, num_labels=2, dropout=0.1):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        hidden_size = self.roberta.config.hidden_size  # 768
        self.dropout = nn.Dropout(dropout)
        # [CLS](768) + 숫자feature(5) → 256 → 2
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + num_features, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels)
        )

    def forward(self, input_ids, attention_mask, num_features, labels=None):
        outputs    = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        combined   = torch.cat([cls_output, num_features], dim=1)
        logits     = self.classifier(combined)
        loss = None
        if labels is not None:
            weight = torch.tensor([2.5, 1.0]).to(labels.device)
            loss   = nn.CrossEntropyLoss(weight=weight)(logits, labels)
        return loss, logits


model_name = 'klue/roberta-small'  # ← 여기만 수정
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = RoBERTaWithFeatures(model_name).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print('✅ 모델 로드 완료')

## 셀 3 — 스케일러 복원 (train.csv 기반)

> 숫자 feature(filler_ratio, vocab_diversity, wpm)를 학습 때와 동일한 분포로 스케일링해야 모델이 올바르게 동작합니다.

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

NUMERIC_COLS = ['total_words', 'filler_count', 'filler_ratio', 'vocab_diversity', 'avg_word_len']

df_train = pd.read_csv(TRAIN_CSV)
scaler   = StandardScaler()
scaler.fit(df_train[NUMERIC_COLS])

print('✅ 스케일러 복원 완료')
print(f'   컬럼: {NUMERIC_COLS}')
print(f'   평균: {dict(zip(NUMERIC_COLS, scaler.mean_.round(4)))}')
print(f'   표준편차: {dict(zip(NUMERIC_COLS, scaler.scale_.round(4)))}')

## 셀 4 — Whisper STT + Feature 추출 함수

### 계산 항목
| Feature | 설명 | 한국어 기준 |
|---|---|---|
| `filler_ratio` | 필러워드 수 / 전체 단어 수 | 낮을수록 ↑ |
| `vocab_diversity` | 고유 단어 수 / 전체 단어 수 (TTR) | 높을수록 ↑ |
| `wpm` | 분당 단어 수 | 평균 약 250~350어절/분 |

In [ ]:
import re
import whisper
import numpy as np

# ── 설정 ──────────────────────────────────────────────
WHISPER_MODEL_SIZE = 'medium'

# 학습 때와 동일한 필러워드 태그 목록
FILLER_TAGS = ['아/', '어/', '음/', '그/', '뭐/', '저/', '이/', 'n/']
# ──────────────────────────────────────────────────────

_whisper_model = None

def get_whisper_model():
    global _whisper_model
    if _whisper_model is None:
        print(f'Whisper {WHISPER_MODEL_SIZE} 모델 로딩 중...')
        _whisper_model = whisper.load_model(WHISPER_MODEL_SIZE)
        print('✅ Whisper 로드 완료')
    return _whisper_model


def clean_text(text: str) -> str:
    """학습(02_preprocessing)과 동일한 방식으로 텍스트 정제"""
    words = text.strip().split()

    # 1) 필러워드 태그 단어 제거
    words = [w for w in words if not any(w.startswith(tag) or w == tag[:-1] for tag in FILLER_TAGS)]
    # 2) 슬래시 포함 단어 제거
    words = [w for w in words if '/' not in w]
    # 3) (N번), (5번) 같은 발화 순서 표기 제거
    words = [w for w in words if not re.match(r'^\((?:\d+번)\)?$', w)]
    # 4) 빈 문자열 제거
    words = [w for w in words if w.strip()]

    return ' '.join(words)


def extract_features(text: str) -> dict:
    """
    학습 때와 동일한 5개 숫자 feature 계산
    - total_words, filler_count, filler_ratio, vocab_diversity, avg_word_len
    """
    words = text.strip().split()
    total_words = len(words)

    # 필러워드 수
    filler_count = sum(1 for w in words if any(w.startswith(tag) or w == tag[:-1] for tag in FILLER_TAGS))
    filler_ratio = filler_count / total_words if total_words > 0 else 0.0

    # clean_words로 vocab_diversity, avg_word_len 계산
    cleaned      = clean_text(text)
    clean_words  = cleaned.split()
    vocab_diversity = len(set(clean_words)) / len(clean_words) if clean_words else 0.0
    avg_word_len    = float(np.mean([len(w) for w in clean_words])) if clean_words else 0.0

    return {
        'total_words':     total_words,
        'filler_count':    filler_count,
        'filler_ratio':    round(filler_ratio, 4),
        'vocab_diversity': round(vocab_diversity, 4),
        'avg_word_len':    round(avg_word_len, 4),
        'clean_text':      cleaned,
    }


def calc_wpm(text: str, duration_seconds: float) -> float:
    """피드백 참고용 WPM 계산 (모델 입력 아님)"""
    total_words = len(text.strip().split())
    if duration_seconds and duration_seconds > 0:
        return round(total_words / (duration_seconds / 60.0), 2)
    return 0.0


def stt_from_audio(audio_path: str) -> tuple[str, float]:
    """Whisper로 음성 → 텍스트 변환, (text, duration_seconds) 반환"""
    wm       = get_whisper_model()
    result   = wm.transcribe(audio_path, language='ko', verbose=False)
    text     = result['text']
    segments = result.get('segments', [])
    duration = segments[-1]['end'] if segments else 0.0
    return text, duration


print('✅ Feature 추출 함수 정의 완료')

## 셀 5 — 추론 + 피드백 출력 함수

In [ ]:
def generate_feedback(features: dict, wpm: float = 0.0) -> dict:
    fb = {}
    fr = features['filler_ratio']
    fc = features['filler_count']
    tw = features['total_words']

    # ── 1) 필러워드 피드백 (3단계) ──────────────────────
    if fr < 0.03:
        fb['filler'] = (
            f"✅ 필러워드 비율 {fr*100:.1f}% — 매우 양호합니다. "
            f"({fc}개 / 전체 {tw}어절)"
        )
    elif fr < 0.07:
        target = int(tw * 0.03)
        reduce = fc - target
        fb['filler'] = (
            f"🟡 필러워드 비율 {fr*100:.1f}% — 보통 수준입니다. "
            f"({fc}개 / 전체 {tw}어절) "
            f"약 {reduce}개 줄이면 3% 이하가 됩니다."
        )
    else:
        target = int(tw * 0.03)
        reduce = fc - target
        fb['filler'] = (
            f"🔴 필러워드 비율 {fr*100:.1f}% — 개선이 필요합니다. "
            f"({fc}개 / 전체 {tw}어절) "
            f"약 {reduce}개 줄여야 3% 이하가 됩니다. "
            f"발표 전 스크립트를 소리내어 연습하며 필러워드를 의식적으로 제거해보세요."
        )

    # ── 2) 어휘 다양성 피드백 (3단계) ──────────────────
    vd = features['vocab_diversity']

    if vd > 0.70:
        fb['vocab'] = (
            f"✅ 어휘 다양성(TTR) {vd:.2f} — 풍부한 어휘를 사용하고 있습니다."
        )
    elif vd > 0.50:
        fb['vocab'] = (
            f"🟡 어휘 다양성(TTR) {vd:.2f} — 보통 수준입니다. "
            f"반복되는 단어를 유사어나 다양한 표현으로 바꿔보세요."
        )
    else:
        fb['vocab'] = (
            f"🔴 어휘 다양성(TTR) {vd:.2f} — 단어 반복이 많습니다. "
            f"같은 표현이 자주 등장해 청중이 지루함을 느낄 수 있습니다. "
            f"핵심 용어 외 일반 단어는 다양하게 바꿔보세요."
        )

    # ── 3) WPM 피드백 (5단계) ──────────────────────────
    if wpm == 0.0:
        fb['wpm'] = "ℹ️ WPM: 음성 길이 정보가 없어 계산할 수 없습니다."
    elif wpm < 200:
        fb['wpm'] = (
            f"🔴 말 속도 {wpm:.0f} WPM — 너무 느립니다. "
            f"한국어 발표 평균(250~350 어절/분)보다 많이 느려 청중이 집중하기 어렵습니다."
        )
    elif wpm < 250:
        fb['wpm'] = (
            f"🟡 말 속도 {wpm:.0f} WPM — 약간 느린 편입니다. "
            f"적정 속도(250~350 어절/분) 하한에 해당합니다. 조금 더 자연스럽게 이어서 말해보세요."
        )
    elif wpm <= 350:
        fb['wpm'] = (
            f"✅ 말 속도 {wpm:.0f} WPM — 적정 속도입니다. "
            f"한국어 발표 평균(250~350 어절/분) 범위에 있습니다."
        )
    elif wpm <= 400:
        fb['wpm'] = (
            f"🟡 말 속도 {wpm:.0f} WPM — 약간 빠른 편입니다. "
            f"강조할 부분에서 의식적으로 속도를 줄여보세요."
        )
    else:
        fb['wpm'] = (
            f"🔴 말 속도 {wpm:.0f} WPM — 너무 빠릅니다. "
            f"한국어 발표 평균(250~350 어절/분)을 크게 초과했습니다. "
            f"문장 사이에 짧은 멈춤을 추가하고 호흡을 조절해보세요."
        )

    return fb


def run_inference(text: str, duration_seconds: float = 0.0) -> dict:
    # 1) feature 추출
    features = extract_features(text)

    # 2) 스케일링
    numeric_raw = np.array([[
        features['total_words'],
        features['filler_count'],
        features['filler_ratio'],
        features['vocab_diversity'],
        features['avg_word_len']
    ]])
    numeric_scaled = scaler.transform(numeric_raw)  # (1, 5)

    # 3) 토크나이징
    encoding = tokenizer(
        features['clean_text'],
        max_length=512,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    # 4) 모델 추론
    input_ids      = encoding['input_ids'].to(DEVICE)
    attention_mask = encoding['attention_mask'].to(DEVICE)
    numeric_tensor = torch.tensor(numeric_scaled, dtype=torch.float32).to(DEVICE)

    with torch.no_grad():
        _, logits = model(input_ids, attention_mask, numeric_tensor)
        probs     = torch.softmax(logits, dim=-1).cpu().numpy()[0]

    pred_idx   = int(np.argmax(probs))
    pred_label = 'Good' if pred_idx == 1 else 'Poor'

    # 5) WPM 계산 (피드백 참고용)
    wpm = calc_wpm(text, duration_seconds)

    # 6) 피드백 생성
    feedback = generate_feedback(features, wpm)

    return {
        'prediction': pred_label,
        'prob_good':  round(float(probs[1]), 4),
        'prob_poor':  round(float(probs[0]), 4),
        'features':   features,
        'wpm':        wpm,
        'feedback':   feedback,
    }


def print_result(result: dict, audio_path: str = None):
    sep = '=' * 60
    print(sep)
    print('  📊 발표 스킬 분석 결과')
    if audio_path:
        print(f'  입력 파일: {audio_path}')
    print(sep)

    pred  = result['prediction']
    emoji = '🟢' if pred == 'Good' else '🔴'
    print(f'\n  최종 판정: {emoji} {pred}')
    print(f'  Good 확률: {result["prob_good"]*100:.1f}%  |  Poor 확률: {result["prob_poor"]*100:.1f}%')

    f = result['features']
    print(f'\n  ─ 수치 요약 ─')
    print(f'  필러워드 비율   : {f["filler_ratio"]*100:.1f}%  ({f["filler_count"]}개 / {f["total_words"]}어절)')
    print(f'  어휘 다양성(TTR): {f["vocab_diversity"]:.3f}')
    print(f'  말 속도(WPM)    : {result["wpm"]:.0f} 어절/분')

    print(f'\n  ─ 세부 피드백 ─')
    for key in ['filler', 'vocab', 'wpm']:
        print(f'  {result["feedback"][key]}')

    print(f'\n{sep}\n')


print('✅ 추론 및 피드백 함수 정의 완료')

## 셀 6 — 음성 파일 입력 → 분석 실행

> `.mp3` 또는 `.wav` 파일 경로를 입력하면 Whisper로 STT 변환 후 자동 분석합니다.

In [ ]:
# ── 음성 파일 경로 설정 ───────────────────────────────
AUDIO_PATH = '/content/drive/MyDrive/presentation_data/my_presentation/발표.mp3'  # 파일명 수정
# ─────────────────────────────────────────────────────

print(f'🎙️  STT 변환 중: {AUDIO_PATH}')
text, duration = stt_from_audio(AUDIO_PATH)

print(f'\n📝 전사 결과 (앞 200자):')
print(text[:200], '...' if len(text) > 200 else '')
print(f'\n⏱️  음성 길이: {duration:.1f}초 ({duration/60:.1f}분)')

result = run_inference(text, duration_seconds=duration)
print_result(result, audio_path=AUDIO_PATH)

## 셀 7 — 텍스트 직접 입력 → 분석 실행

> 음성 파일 없이 스크립트를 직접 붙여넣어 분석할 때 사용합니다.  
> WPM 계산을 위해 발표 시간(분)을 함께 입력하세요.

In [ ]:
# ── 텍스트 및 발표 시간 입력 ─────────────────────────
SCRIPT = """
여기에 발표 스크립트를 붙여넣으세요.
""".strip()

DURATION_MINUTES = 3.0   # 발표 시간(분). 모르면 None
# ─────────────────────────────────────────────────────

duration_sec = DURATION_MINUTES * 60 if DURATION_MINUTES else 0.0

result = run_inference(SCRIPT, duration_seconds=duration_sec)
print_result(result)